# task 1 for lab including set up

## The set up area

- effectively this bit is to set up the imports, as well as the tables needed for the tasks
- all of the code written here are based off of kokchuns examples in his lectures, I've just written them all down in a notebook to look and rewrite on a computer afterwards
- I want to make the note here that I have not used AI or LLMs at all during this, all of this is hand programmed by me!

In [1]:
import duckdb

from pathlib import Path

import matplotlib as plot


- calling for the tables up here means that I can quickly run them to check if they have been found and properly registered as their new variable

In [2]:
duckdb_path = "Data/sakila.duckdb"
Path(duckdb_path).unlink(missing_ok=True)
with duckdb.connect(duckdb_path) as conn, open("sql/load_sakila.sql") as ingest_scripts: 
    conn.sql(ingest_scripts.read())
    
    descrition = conn.sql("DESC;").df()
    films = conn.sql("FROM film;").df()
    actors = conn.sql("FROM actor;").df()
    film_actors = conn.sql("FROM film_actor").df()
    customers = conn.sql("FROM customer;").df()
    payments = conn.sql("FROM payment;").df()
    categories = conn.sql("FROM category;").df()
    film_categories = conn.sql("FROM film_category;").df()
    inventory = conn.sql("FROM inventory;").df()
    rentals = conn.sql("FROM rental;").df()
    

## task 1a) Movie length over 3h + task 1f) shortest movies
- this task was simple really, basically call for the title column from the film table
- ask for it to select all of the titles where the length column was over 180 as its showing the time in min (basic math 3h is 180mins)
- order it by descending, so it starts with the longest films first no matter what which are all 185 
- limiting it down to 15 becuase there is a total of 39 films that are over 180 min

In [ ]:
duckdb.sql("""--sql
           SELECT title,
           length 
           FROM films
           WHERE length > 180
           ORDER BY length DESC
           LIMIT 15
           """)            

- this bit here is the first question for 1f so I figured that 1f should always be the opposite to what the original question was, so in this case what is the shortest film available?
- so I did the same steps as I did in 1a, and flipped the script, instead of looking for films over 180min, I looked under 60min to find the shortest
- order it by ascending, this time to start from our shortest films of 46 mins
- still limiting it because there are too many rows, 96 to be more exact, so limiting it down to 20 rows

In [ ]:
duckdb.sql("""--sql
           SELECT title,
           length
           FROM films
           WHERE length < 60
           ORDER BY length ASC
           LIMIT 20
           """)

## task 1b) Movies with Love in title + task 1f) Movies with Hate in title
- the next task, 1b, was a bit more complicated to do, maybe because I forgot what a whildcard was for a while
- I was asked to find film titles with the word 'love' in their title, as a stand alone word not combined or the expressive term
- with the added notes of needing to show not only title but to also show, the films rating, length and description
- so I started the same as in task 1a, calling for the columns I needed from the film table, then using the LIKE operator together with a wildcard to find the word 'love'
- with this all done we find 4 films with the word love in the title of them

In [ ]:
duckdb.sql("""--sql
           SELECT title,
           rating,
           length,
           description,
           FROM films
           WHERE 
           title LIKE '%LOVE'
            """)

- so task 1f here is of course loves true oppisition hate
- did the same as task 1b, and replaced 'love' with 'hate'
- which suprisingly enough actually works because there is only one film with the word 'hate' in the title

In [ ]:
duckdb.sql("""--sql
           SELECT title,
           rating,
           length,
           description
           FROM films
           WHERE
           title LIKE '%HATE'
           """)

## 1c) Avarage movie length
- next up, task 1c, what is the avarage movie length/median movie length at this rental store?
- so how did I approach, well I called for the length column from the film table, then I aggregated it to get the values needed to awnser the questions
- what is the shortest film? 46 mins
- what is the longest film? 185 mins(3h5m)
- what is the avarage film length? 115.272 mins(~1h55m)
- what is the median film length? 114 mins(1h54m)

In [ ]:
duckdb.sql("""--sql
           SELECT 
           MIN (length) AS Short_movie,
           MAX (length) AS Long_movie,
           AVG (length) AS avg_movie_length,
           MEDIAN (length) AS median_movie
           FROM films""")

## 1d) top 10 expensive rental + 1f) top 10 cheapest 
- similar to how I did the others, I did this one easily, called for titles, rental rate and duration from the film table
- ordered it by descending and limiting it to 10 as was asked for, so it starts with the expensive films to rent, for 5

In [ ]:
duckdb.sql("""--sql
           SELECT title,
           rental_rate,
           rental_duration
           FROM films
           ORDER BY rental_rate DESC
           LIMIT 10""")

- and following up with the second to last bit of 1f, doing the opposite
- just ordering it by ascending now so it starts at 1

In [ ]:
duckdb.sql("""--sql
           SELECT title,
           rental_rate,
           rental_duration
           FROM films
           ORDER BY rental_rate ASC
           LIMIT 10""")

## 1e) most movies for one actor + 1f) the least movies for one actor
- now for the last set of questions relating to task 1 for the lab
- what actor has appeared in most films? and what actor has apppeared the least in films?
- starting with a join now instead
- joined together the film, actor and film actor tables via a left join to get the left sided columns of all tables

In [ ]:
films_joined = duckdb.sql("""--sql
                          SELECT *
                          FROM films f
                          LEFT JOIN film_actors fa ON f.film_id = fa.film_id
                          LEFT JOIN actors a ON a.actor_id = fa.actor_id""").df()

films_joined #checking to make sure it all worked

- then after the join I started calling for the columns needed from the tables
- counted the number of appearences in films, then used group by, followed by order by and then limiting it to the top 10 of most appearences

In [ ]:
duckdb.sql("""--sql
           SELECT 
           actor_id,
           first_name,
           last_name,
           COUNT(*) AS number_films
           FROM 
           films_joined
           GROUP BY 
           first_name,
           actor_id, 
           last_name
           ORDER BY number_films DESC
           LIMIT 10
           """)

- then ending task 1f, with the opposite of the least amount of appearences, just ordered by ascending so its the bottom 10

In [ ]:
duckdb.sql("""--sql
           SELECT
           actor_id,
           first_name,
           last_name,
           COUNT(*) AS number_films
           FROM
           films_joined
           GROUP BY 
           first_name,
           last_name,
           actor_id
           ORDER BY number_films ASC
           LIMIT 10""")

# TASK 2 AREA
- this whole bit is for task 2 in the lab, which is playing around with displaying graphs

## task 2a) top 5 customers
- starting off with a left join to get the customer table and the payment table together

In [6]:
customer_spend = duckdb.sql("""--sql
           SELECT *
           FROM customers c
           LEFT JOIN payments p ON c.customer_id = p.customer_id
           """).df()


- then making the amounts spent as its own seperate variable to make the call for the graph easier
- counting it all then ordering by descending on amount of money spent

In [17]:
amounts_spent = duckdb.sql("""--sql
           SELECT
           customer_id,
            email,
           COUNT(*) AS money_spent
           FROM
           customer_spend
           GROUP BY
           customer_id,
            email
           ORDER BY money_spent DESC
            """).df()

- this displays the graph for the money that was spent by the top 5 customers, limiting it to 5, because we just want the top 5

In [ ]:
ax = amounts_spent.head(5).plot(
    kind ="barh",
    x ="customer_id",
    y = "money_spent",
    title = "our top 5 customers",
    xlabel = "amount spent at us")
ax.invert_yaxis()

- this is here to quickly get the email connected to the costumer ids so that we can send our rewards to the best costumers

In [ ]:
amounts_spent.head(5)

## task 2b) our most profitible film categories
- starting similarly to 2a with a join of all the tables needed
- look there is no better name than that end of discussion

In [20]:
theJOIN_monster = duckdb.sql("""--sql
           SELECT *
           FROM categories cat
           FULL JOIN film_categories fc ON cat.category_id = fc.category_id
            FULL JOIN films f ON f.film_id = fc.film_id
            FULL JOIN inventory i ON i.film_id = fc.film_id
            FULL JOIN rentals r ON r.inventory_id = i.inventory_id
            FULL JOIN payments p ON p.rental_id = r.rental_id
           """).df()

- now working our way to the category revenue, doing the same as before
- calling for the column needed, counting the revenue, ordering it by descending so we get the top results

In [22]:
category_revenue = duckdb.sql("""--sql
           SELECT
           category_id,
           COUNT(*) AS category_revenue
           FROM
           theJOIN_monster
           GROUP BY
           category_id
           ORDER BY category_revenue DESC
           """).df()

- ending with the graph for our categories, unfortunatly I can't get it to show category names, only ids, so we're going to have to ask theJOIN_monster for the names

In [ ]:
ax = category_revenue.plot(
    kind ="barh",
    x ="category_id",
    y = "category_revenue",
    title = "our categories(id)",
    xlabel = "amount from the categories")
ax.invert_yaxis()